# Deep Q-Learning pour un robot d'entrepot

Ce notebook implemente et entraine un agent **Deep Q-Network (DQN)** dans un environnement Gymnasium simplifie.

Objectif : recuperer des articles, eviter les obstacles et les deposer dans la zone de livraison.

## Installation

Depuis la racine du projet, installez les dependances :

```bash
pip install -r requirements.txt
```

Puis demarrez Jupyter avec :

```bash
jupyter lab
```

In [ ]:
from pathlib import Path
import sys

# Le notebook est dans notebooks/ : on ajoute la racine du projet aux imports.
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np

from warehouse_env import WarehouseEnv
from dqn_agent_tf import DQNAgent, train


## Le MDP de l'entrepot

- **Etat** : position du robot, information sur le transport d'un article et positions des articles restants.
- **Actions** : haut, bas, gauche, droite, saisir, deposer.
- **Recompenses** : cout faible par mouvement, bonus pour saisir/deposer, penalite de collision et bonus final.
- **Objectif** : maximiser la somme des recompenses futures.

In [ ]:
env = WarehouseEnv(grid_size=6, n_items=1, max_steps=80)
state, info = env.reset(seed=42)

print('Etat initial :', state)
print('Taille de l etat :', env.observation_space.shape[0])
print('Nombre d actions :', env.action_space.n)
print()
print(env.render())

### Legende

- `R` : robot sans article
- `C` : robot transportant un article
- `I` : article a recuperer
- `D` : zone de depot
- `X` : obstacle
- `.` : cellule libre

## Creation de l'agent DQN

Le reseau recoit un etat en entree et retourne une Q-value par action. L'agent choisit generalement l'action avec la Q-value la plus elevee, tout en gardant une part d'exploration avec la strategie epsilon-greedy.

In [ ]:
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = DQNAgent(
    state_size=state_size,
    action_size=action_size,
    learning_rate=1e-3,
    gamma=0.99,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
    memory_size=20_000,
    batch_size=64,
)

print('Agent DQN initialise.')

## Entrainement

Pour une premiere demonstration, l'environnement reste volontairement petit : une grille 6x6 et un seul article. Augmentez progressivement la complexite apres avoir observe une convergence fiable.

In [ ]:
history = train(
    env=env,
    agent=agent,
    episodes=1500,
    max_steps=80,
    target_update_every=25,
)

## Analyse des resultats

Une moyenne mobile aide a distinguer la tendance generale de la variabilite naturelle entre les episodes.

In [ ]:
window = 50
rewards = np.asarray(history['rewards'])
delivered = np.asarray(history['delivered'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(rewards, alpha=0.25, label='Par episode')
axes[0].plot(range(window - 1, len(rewards)),
             np.convolve(rewards, np.ones(window) / window, mode='valid'),
             linewidth=2, label='Moyenne mobile (50)')
axes[0].set_title('Recompense totale')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Recompense')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(delivered, alpha=0.25, label='Par episode')
axes[1].plot(range(window - 1, len(delivered)),
             np.convolve(delivered, np.ones(window) / window, mode='valid'),
             linewidth=2, label='Moyenne mobile (50)')
axes[1].set_title('Articles livres')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Nombre livre')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Evaluation sans exploration

L'evaluation desactive l'exploration aleatoire : l'agent utilise uniquement les Q-values apprises pour selectionner ses actions.

In [ ]:
evaluation_env = WarehouseEnv(grid_size=6, n_items=1, max_steps=80)
state, _ = evaluation_env.reset(seed=7)
total_reward = 0.0

for step in range(80):
    action = agent.act(state, training=False)
    state, reward, terminated, truncated, info = evaluation_env.step(action)
    total_reward += reward

    print(f'Etape {step + 1} | action={action} | reward={reward:.1f}')
    print(evaluation_env.render())
    print()

    if terminated or truncated:
        break

print(f'Recompense finale : {total_reward:.2f}')
print(f'Articles livres : {info["delivered"]}')

## Prochaines experimentations

1. Augmenter progressivement le nombre d'articles et d'obstacles.
2. Normaliser les coordonnees des etats entre 0 et 1.
3. Ajouter du bruit aux mouvements ou aux detections de capteurs.
4. Comparer DQN, Double DQN et Dueling DQN.
5. Remplacer les coordonnees par une image de grille et utiliser un CNN.